In [3]:
from pathlib import Path
import pandas as pd

df = pd.read_csv('scores.csv')

required = {"student_id", "name", "midterm", "final"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"缺少列: {sorted(missing)}")

if df["student_id"].duplicated().any():
    raise ValueError("student_id 必须唯一")

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.head())

(8, 5)
student_id        str
name              str
quiz            int64
midterm         int64
final         float64
dtype: object
student_id    0
name          0
quiz          0
midterm       0
final         1
dtype: int64
  student_id name  quiz  midterm  final
0       S001   陈晨    82       76   88.0
1       S002   李雷    65       58   72.0
2       S003   王芳    91       89   94.0
3       S004   赵敏    55       63   59.0
4       S005   周航    78       70   85.0


In [4]:
score_cols = ["quiz", "midterm", "final"]
scores = df[score_cols].to_numpy(dtype=float, copy=True)
scores

array([[82., 76., 88.],
       [65., 58., 72.],
       [91., 89., 94.],
       [55., 63., 59.],
       [78., 70., 85.],
       [88., 92., 90.],
       [70., 66., nan],
       [60., 50., 65.]])

第 1 题：销售表读取与数组统计
要求：

用 StringIO 和 pd.read_csv 读取，不得落地临时文件。
校验 order_id 唯一，月份列必须能转为数值且非缺失值位于 [0, +∞)。
将 jan/feb/mar 转为 NumPy 数组。
为每个订单计算 quarter_total，缺失月份按 0 计。
计算各月均值，均值忽略缺失值，返回索引为 jan/feb/mar 的 Series。

In [15]:
from io import StringIO
import numpy as np
csv_text="""
order_id,region,jan,feb,mar
O01,East,120,135,150
O02,West,90,110,105
O03,East,200,,220
O04,North,80,95,100
"""
def sales_summary(csv_text: str) -> tuple[pd.DataFrame, pd.Series]:
    df=pd.read_csv(StringIO(csv_text))
    if (df['order_id'].duplicated().any()) | ((df[['jan','feb','mar']]<0).any().any()):
        raise ValueError("订单号必须唯一，且月份列必须能转为数值且非缺失值位于 [0, +∞)")
    Jan=df['jan'].to_numpy(dtype=float,copy=True)
    Feb=df['feb'].to_numpy(dtype=float,copy=True)
    Mar=df['mar'].to_numpy(dtype=float,copy=True)
    result=pd.Series([np.nanmean(Jan),np.nanmean(Feb),np.nanmean(Mar)],index=['jan','feb','mar'])
    np.nan_to_num(Jan,nan=0,copy=False)
    np.nan_to_num(Feb,nan=0,copy=False)
    np.nan_to_num(Mar,nan=0,copy=False)
    quarter_total=Jan+Feb+Mar
    df=df.assign(quarter_total=quarter_total)
    return df,result
sales_summary(csv_text)

(  order_id region  jan    feb  mar  quarter_total
 0      O01   East  120  135.0  150          405.0
 1      O02   West   90  110.0  105          305.0
 2      O03   East  200    NaN  220          420.0
 3      O04  North   80   95.0  100          275.0,
 jan    122.500000
 feb    113.333333
 mar    143.750000
 dtype: float64)

第 2 题：传感器有效率与告警行
给定数组，每行是一台设备，每列依次为 08:00、12:00、16:00 的温度；NaN 表示未上报：
规则：

有效率 = 非 NaN 数量 / 3。
平均温度忽略 NaN；全缺失设备的平均温度为 NaN。
只要任一有效温度大于等于 30℃，alert=True。
返回列：device_id, valid_rate, mean_temp, alert，顺序与输入一致。

In [21]:
device_ids = np.array(["D1", "D2", "D3", "D4"])
temps = np.array([
    [22.0, 25.0, 27.0],
    [31.0, np.nan, 35.0],
    [18.0, 19.0, 20.0],
    [np.nan, np.nan, np.nan],
])
def sensor_report(device_ids: np.ndarray, temps: np.ndarray) -> pd.DataFrame:
    df=pd.DataFrame(device_ids,columns=['device_id'])
    df['valid_rate']=np.sum(~np.isnan(temps),axis=1)/3
    df['mean_temp']=np.nanmean(temps,axis=1)
    df['alert']=np.any(temps>=30,axis=1)
    return df
sensor_report(device_ids,temps)

C:\Users\Qiutian\AppData\Local\Temp\ipykernel_43436\1594595465.py:11: RuntimeWarning: Mean of empty slice
  df['mean_temp']=np.nanmean(temps,axis=1)


,device_id,valid_rate,mean_temp,alert
0,D1,1.000000,24.666667,False
1,D2,0.666667,33.000000,True
2,D3,1.000000,19.000000,False
3,D4,0.000000,NaN,False


第 3 题：成绩加权、资格筛选与稳定排名
输入 DataFrame 包含 student_id, name, quiz, midterm, final。编写函数：
规则：

校验学号唯一、三项成绩均为 0–100 的数值。
任一成绩缺失者 eligible=False，加权分为 NaN。
三项齐全者按 quiz*0.2 + midterm*0.3 + final*0.5 计算 weighted_score。
资格学生按加权分降序，同分按 student_id 升序；无资格学生置后，也按学号升序。
返回新 DataFrame，不得修改输入对象。

In [27]:
def rank_students(df: pd.DataFrame) -> pd.DataFrame:
    if (df['student_id'].duplicated().any())|((df[['quiz','midterm','final']]<0).any().any())|((df[['quiz','midterm','final']]>100).any().any()):
        raise ValueError("学号必须唯一，且三项成绩均为 0–100 的数值")
    df['eligible']=df[['quiz','midterm','final']].notna().all(axis=1)
    df['weighted_score']=0.2*df['quiz']+0.3*df['midterm']+0.5*df['final']
    df=df.sort_values(
        ['weighted_score','student_id','eligible'],
        ascending=[False,True,False],
    )
    return df
df = pd.DataFrame({
    "student_id": ["S03", "S01", "S04", "S02"],
    "name": ["丙", "甲", "丁", "乙"],
    "quiz": [80, 90, 70, 90],
    "midterm": [80, 80, 75, 80],
    "final": [80, 80, np.nan, 80],
})  
rank_students(df)

,student_id,name,quiz,midterm,final,eligible,weighted_score
1,S01,甲,90,80,80.0,True,82.0
3,S02,乙,90,80,80.0,True,82.0
0,S03,丙,80,80,80.0,True,80.0
2,S04,丁,70,75,NaN,False,NaN
